# Notebook 02: Train SPAGHETTI-1 models on Kaggle (2 x T4)

This notebook trains a list of experiment configs from the team repository on Kaggle's free GPUs and exports the results as two zip files you can download. Full guide: `docs/11_KAGGLE_TRAINING.md`.

**Before you run it** (right-hand panel, *Session options*):

1. **Accelerator: GPU T4 x2.**
2. **Internet: on.** This needs a phone-verified Kaggle account. It is used to clone the repository and download the pretrained weights.
3. **Add Input → Competitions → `an2dl2627`** (the official data; nothing needs to be uploaded).
4. Only if the repository is private: *Add-ons → Secrets* → add your GitHub token as `GITHUB_PAT`.

For long runs use **Save Version → Save & Run All (Commit)**. The notebook then runs in the background, also with the browser closed, for up to 12 hours. The zip files appear under *Output* when it finishes.

## Step 1: Get the code and two packages

Kaggle already has PyTorch, OpenCV, scikit-learn, and pandas. `timm` is upgraded because DINOv3 needs timm 1.0.20 or newer.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "github.com/AN2DL-Challenge-26-27/AN2DL-Challenge-26-27.git"
REPO_DIR = Path("/kaggle/working/AN2DL-Challenge-26-27")

if not REPO_DIR.exists():
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_PAT")
    except Exception:
        token = None
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["git", "clone", "--depth", "1", url, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "timm>=1.0.22", "python-dotenv"], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
import timm, torch
print("repository:", REPO_DIR, "| commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
print("torch", torch.__version__, "| timm", timm.__version__)
for gpu in range(torch.cuda.device_count()):
    print(f"GPU {gpu}: {torch.cuda.get_device_name(gpu)} | bf16 supported: {torch.cuda.is_bf16_supported()}")

## Step 2: Find the competition data

The code reads the raw competition files directly and uses the team's manifests and folds from the repository (`manifests/`), so the results are comparable with every earlier run.

In [ ]:
sys.path.insert(0, str(REPO_DIR / "experiments"))
from kaggle_queue import find_data_dir

DATA_DIR = find_data_dir()
print("data:", DATA_DIR, "|", len(list((DATA_DIR / "train").glob("*.png"))), "train /", len(list((DATA_DIR / "test").glob("*.png"))), "test images")

## Step 3: Choose what to train

Pick one plan, or write your own list of config files from `configs/experiments/`. The configs are spread over both GPUs. Rough durations on 2 x T4 (to be confirmed on the first run):

| Plan | Runs | Approximate time |
| --- | --- | --- |
| `dinov2_5fold` | DINOv2-Base 336, folds 1–4 (fold 0 already trained on the laptop) | 2–3 h |
| `dinov2_5fold_all` | DINOv2-Base 336, folds 0–4 | 3–4 h |
| `dinov3_5fold` | DINOv3 ViT-B/16 384, folds 0–4 | 3–4 h |
| `custom` | your own `CUSTOM_CONFIGS` list | |

Every run also predicts the 14,736 test images (without TTA, which did not help DINOv2).

In [ ]:
PLAN = "dinov2_5fold"          # dinov2_5fold | dinov2_5fold_all | dinov3_5fold | custom
CUSTOM_CONFIGS = []             # e.g. ["configs/experiments/n1_dinov3_convnext_tiny_288_fold0.env"]

PLANS = {
    "dinov2_5fold": [f"configs/experiments/r2_dinov2_base_336_recipe_fold{k}.env" for k in (1, 2, 3, 4)],
    "dinov2_5fold_all": [f"configs/experiments/r2_dinov2_base_336_recipe_fold{k}.env" for k in range(5)],
    "dinov3_5fold": [f"configs/experiments/n2_dinov3_base_384_recipe_fold{k}.env" for k in range(5)],
    "custom": CUSTOM_CONFIGS,
}
CONFIGS = PLANS[PLAN]
assert CONFIGS, "empty plan"
for config in CONFIGS:
    assert (REPO_DIR / config).is_file(), f"missing {config}"
print(f"{len(CONFIGS)} runs:")
for config in CONFIGS:
    print("  ", config)
# Inspect the full settings of the first run (no training):
subprocess.run([sys.executable, "-m", "spaghetti1", "train", "--config", CONFIGS[0], "--dry-run"])

## Step 4: Train

Progress is printed every minute: the current run on each GPU and its latest epoch. The full logs are in `/kaggle/working/runs/logs/`. A run that fails does not stop the others. If the session is interrupted, running this cell again continues unfinished runs (`RESUME=true`).

In [ ]:
TAG = f"kaggle_{PLAN}"
command = [sys.executable, "-u", "experiments/kaggle_queue.py", "--configs", *CONFIGS,
           "--data-dir", str(DATA_DIR), "--work-dir", "/kaggle/working", "--tag", TAG]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
print("exit code:", process.wait())

## Step 5: Results

`summary/summary.json` lists every run, its status, and for each group of folds the per-fold and **out-of-fold** weighted F1 (about 3,000 images, more reliable than one fold). It also contains a fold-ensemble submission (average of the folds' test probabilities).

In [ ]:
import json
import pandas as pd

summary = json.loads(Path("/kaggle/working/summary/summary.json").read_text())
print(json.dumps(summary["fold_groups"], indent=2))
results = pd.read_csv("/kaggle/working/runs/results.csv")
results[["run_name", "fold", "image_size", "best_epoch", "weighted_f1", "weighted_f1_tta", "minutes"]]

## Step 6: Download and share the results

Two files are written to `/kaggle/working/` (shown under *Output* after a committed run):

| File | Contents | Use |
| --- | --- | --- |
| `spaghetti1_results_<tag>.zip` | Everything except model weights: results, logs, configs, curves, validation and test probabilities, submissions, summary | Small: send this one for analysis and ensembling |
| `spaghetti1_weights_<tag>.zip` | The trained models (`best.pt`) with checksums | Large: upload to the team Drive; `python -m spaghetti1 fetch-weights --source <zip>` |

A fold-ensemble `submission.csv` can be submitted directly from `summary/fold_ensemble_<group>/`.

In [ ]:
from IPython.display import FileLink, display

for path in sorted(Path("/kaggle/working").glob("spaghetti1_*.zip")):
    print(f"{path.name}: {path.stat().st_size / 2**20:.0f} MB")
    display(FileLink(str(path.relative_to("/kaggle/working"))))
for path in sorted(Path("/kaggle/working/summary").glob("fold_ensemble_*/submission.csv")):
    print("fold-ensemble submission:", path)